# EDA and Modeling with PCA & K-Fold Validation (Leakage Fixed)
This notebook handles the data pipeline: loading the dataset, performing train-test split first to prevent data leakage, conducting exploratory data analysis, selecting the top features, and training/evaluating models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.decomposition import PCA
sns.set_theme(style="whitegrid")

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/divorce.csv', sep=';')
if len(df.columns) == 1:
    df = pd.read_csv('../data/divorce.csv', sep=',')
df.dropna(inplace=True)
if 'Id' in df.columns:
    df.drop('Id', axis=1, inplace=True)

## 2. Train/Test Split (FIRST to avoid Data Leakage)

In [ ]:
X_all = df.drop('Class', axis=1)
y = df['Class']
X_train_all, X_test_all, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

## 3. Exploratory Data Analysis (on full dataset is fine for visual exploration)

In [ ]:
plt.figure(figsize=(20, 15))
df.hist(bins=15, figsize=(20, 15), layout=(8, 7))
plt.tight_layout()
plt.show()

### PCA to visually prove linearly separable dataset

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_all)
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette='Set1', s=100)
plt.title('PCA of Divorce Dataset (Showing Perfect Separability)')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.show()

## 4. Feature Selection (on TRAIN SET ONLY to prevent leakage)

In [ ]:
train_df = pd.concat([X_train_all, y_train], axis=1)
corr_matrix_train = train_df.corr()
plt.figure(figsize=(20, 15))
sns.heatmap(corr_matrix_train, cmap='coolwarm', annot=False, fmt=".2f")
plt.title("Correlation Heatmap (Train Set Only)")
plt.show()

### Select top 10 features from train set only

In [ ]:
top_features = corr_matrix_train['Class'].sort_values(ascending=False).head(11).index.tolist()
top_features.remove('Class')
X_train_top = X_train_all[top_features]
X_test_top = X_test_all[top_features]

## 5. Model Training (Logistic Regression)

In [ ]:
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_all, y_train)

optimized_model = LogisticRegression(max_iter=1000)
optimized_model.fit(X_train_top, y_train)

## 6. K-Fold Cross Validation (on train set top features)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(optimized_model, X_train_top, y_train, cv=cv, scoring='accuracy')
print(f"5-Fold CV Accuracy (Train Set): {scores}")
print(f"Mean CV Accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

## 7. Evaluation on Unseen Test Set

In [ ]:
y_pred_base = baseline_model.predict(X_test_all)
y_pred_opt = optimized_model.predict(X_test_top)

print("Baseline (All Features) Metrics:", accuracy_score(y_test, y_pred_base), precision_score(y_test, y_pred_base), recall_score(y_test, y_pred_base))
print("Optimized (Top Features) Metrics:", accuracy_score(y_test, y_pred_opt), precision_score(y_test, y_pred_opt), recall_score(y_test, y_pred_opt))

In [ ]:
cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (Optimized Model on Test Set)')
plt.show()

## 8. Export Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(optimized_model, '../models/logistic_model.pkl')
joblib.dump(top_features, '../models/top_features.pkl')